Implement both policy iteration and value iteration algorithms using dynamic programming. Apply them to a given MDP and compare their performance in finding the optimal policy.

In [1]:
import numpy as np

N, GAMMA, THETA = 4, 0.9, 0.001
START, GOAL = (0, 0), (3, 3)
actions = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}
symbols = {0: "↑", 1: "↓", 2: "←", 3: "→"}

In [2]:
def step(state, action):
    if state == GOAL:
        return GOAL, 0
    r, c = state
    dr, dc = actions[action]
    nr = min(max(r + dr, 0), N - 1)
    nc = min(max(c + dc, 0), N - 1)
    next_state = (nr, nc)
    return next_state, (1 if next_state == GOAL else -1)

In [3]:
def get_q_values(state, V):
    q_values = []
    for action in actions:
        (nr, nc), reward = step(state, action)
        q_values.append(reward + GAMMA * V[nr, nc])
    return q_values

In [4]:
def policy_evaluation(policy):
    V = np.zeros((N, N))
    while True:
        delta = 0
        for r in range(N):
            for c in range(N):
                if (r, c) != GOAL:
                    (nr, nc), reward = step((r, c), policy[r, c])
                    new_v = reward + GAMMA * V[nr, nc]
                    delta = max(delta, abs(new_v - V[r, c]))
                    V[r, c] = new_v
        if delta < THETA:
            return V

In [5]:
def policy_iteration():
    policy = np.random.randint(0, 4, size=(N, N))
    policy[GOAL] = -1
    iteration = 0
    while True:
        V = policy_evaluation(policy)
        stable = True
        for r in range(N):
            for c in range(N):
                if (r, c) != GOAL:
                    best_action = np.argmax(get_q_values((r, c), V))
                    if policy[r, c] != best_action:
                        stable = False
                    policy[r, c] = best_action
        iteration += 1
        print("Policy Iteration:", iteration)
        if stable:
            return V, policy, iteration

In [6]:
def value_iteration():
    V = np.zeros((N, N))
    iteration = 0
    while True:
        delta = 0
        new_V = V.copy()
        for r in range(N):
            for c in range(N):
                if (r, c) != GOAL:
                    best_v = max(get_q_values((r, c), V))
                    delta = max(delta, abs(best_v - V[r, c]))
                    new_V[r, c] = best_v
        V = new_V
        iteration += 1
        print("Value Iteration:", iteration)
        if delta < THETA:
            break

    policy = np.full((N, N), -1)
    for r in range(N):
        for c in range(N):
            if (r, c) != GOAL:
                policy[r, c] = np.argmax(get_q_values((r, c), V))
    return V, policy, iteration

In [7]:
def display_results(title, V, policy, iterations):
    print("\nOptimal Value Function:")
    for row in V:
        print(" ".join(f"{value:7.3f}" for value in row))
    print("\nOptimal Policy:")
    for r in range(N):
        row = ["S" if (r, c) == START else ("G" if (r, c) == GOAL else symbols[policy[r, c]]) for c in range(N)]
        print("  ".join(row))
    print("\nTotal Iterations:", iterations)

V_pi, policy_pi, iter_pi = policy_iteration()
display_results("POLICY ITERATION", V_pi, policy_pi, iter_pi)

V_vi, policy_vi, iter_vi = value_iteration()
display_results("VALUE ITERATION", V_vi, policy_vi, iter_vi)

print("\n==============================\nCOMPARISON\n==============================")
print("Policy Iteration took", iter_pi, "iterations")
print("Value Iteration took", iter_vi, "iterations")
print("Same Optimal Policy:", np.array_equal(policy_pi, policy_vi))

Policy Iteration: 1
Policy Iteration: 2
Policy Iteration: 3
Policy Iteration: 4
Policy Iteration: 5

Optimal Value Function:
 -3.505  -2.783  -1.981  -1.090
 -2.783  -1.981  -1.090  -0.100
 -1.981  -1.090  -0.100   1.000
 -1.090  -0.100   1.000   0.000

Optimal Policy:
S  ↓  ↓  ↓
↓  ↓  ↓  ↓
↓  ↓  ↓  ↓
→  →  →  G

Total Iterations: 5
Value Iteration: 1
Value Iteration: 2
Value Iteration: 3
Value Iteration: 4
Value Iteration: 5
Value Iteration: 6
Value Iteration: 7

Optimal Value Function:
 -3.505  -2.783  -1.981  -1.090
 -2.783  -1.981  -1.090  -0.100
 -1.981  -1.090  -0.100   1.000
 -1.090  -0.100   1.000   0.000

Optimal Policy:
S  ↓  ↓  ↓
↓  ↓  ↓  ↓
↓  ↓  ↓  ↓
→  →  →  G

Total Iterations: 7

COMPARISON
Policy Iteration took 5 iterations
Value Iteration took 7 iterations
Same Optimal Policy: True
